# Sport Action Recognition CNN3D
Classification de 3 actions sportives (JumpingJack, Punch, SoccerJuggling) 
sur le dataset UCF101 avec un CNN 3D.
Test accuracy : 88.89%

In [3]:
import os
import shutil

BASE_PATH = '/kaggle/input/datasets/matthewjansen/ucf101-action-recognition'
CLASSES = ['JumpingJack', 'Punch', 'SoccerJuggling']
WORK_DIR = '/kaggle/working/dataset'

for split in ['train', 'test']:
    for cls in CLASSES:
        src = os.path.join(BASE_PATH, split, cls)
        dst = os.path.join(WORK_DIR, split, cls)
        if os.path.exists(src):
            shutil.copytree(src, dst)
            count = len(os.listdir(dst))
            print(f"{split}/{cls} : {count} vidéos")
        else:
            print(f"MANQUANT : {split}/{cls}")
            

train/JumpingJack : 92 vidéos
train/Punch : 120 vidéos
train/SoccerJuggling : 110 vidéos
test/JumpingJack : 16 vidéos
test/Punch : 20 vidéos
test/SoccerJuggling : 19 vidéos


In [4]:
!pip install -q tensorflow opencv-python-headless

import tensorflow as tf
import cv2
import numpy as np
import os
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder

In [5]:
import os
X = []
y = []
CLASSES = ['JumpingJack', 'Punch', 'SoccerJuggling']

for classe in CLASSES:
    dossier = f'/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/{classe}'
    videos = os.listdir(dossier)
    for video in videos:
        chemin = os.path.join(dossier, video)
        obj = cv2.VideoCapture(chemin)
        nbFrame = obj.get(cv2.CAP_PROP_FRAME_COUNT)
        array = np.linspace(0, nbFrame - 1, 16)
        frame_video = []
        for nb in array:
            obj.set(cv2.CAP_PROP_POS_FRAMES, nb)
            (ret, frame) = obj.read()
            
            if ret:
                frame = cv2.resize(frame, (112, 112))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame/255
                frame_video.append(frame)
        if len(frame_video) == 16:
            X.append(frame_video)
            y.append(classe)
X = np.array(X, dtype=np.float32)
print("Shape X:", X.shape)
print("Shape y:", len(y))

Shape X: (315, 16, 112, 112, 3)
Shape y: 315


In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [7]:
from tensorflow.keras import layers, models

def build_model(num_classes=3):
    model = models.Sequential([
        layers.Conv3D(32, kernel_size=(3, 3, 3), activation='relu', padding='same',
                      input_shape=(16, 112, 112, 3)),
        layers.MaxPooling3D(pool_size=(1, 2, 2)),
        layers.Conv3D(64, kernel_size=(3, 3, 3), activation='relu', padding='same'),
        layers.MaxPooling3D(pool_size=(2, 2, 2)),
        layers.Conv3D(128, kernel_size=(3, 3, 3), activation='relu', padding='same'),
        layers.MaxPooling3D(pool_size=(2, 2, 2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 16, 112, 112,   │         2,624 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 16, 56, 56, 32) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 16, 56, 56, 64) │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 8, 28, 28, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 8, 28, 28, 128) │       221,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_2 (MaxPooling3D)  │ (None, 4, 14, 14, 128) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,970,435 (99.07 MB)

 Trainable params: 25,970,435 (99.07 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
from tensorflow.keras.utils import to_categorical
y_cat = to_categorical(y_encoded, num_classes=3)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X, y_cat,
    epochs=15,
    batch_size=8,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 557ms/step - accuracy: 0.6270 - loss: 1.0521 - val_accuracy: 0.8571 - val_loss: 0.9436
Epoch 2/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 155ms/step - accuracy: 0.7579 - loss: 0.5538 - val_accuracy: 0.4286 - val_loss: 0.9669
Epoch 3/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 156ms/step - accuracy: 0.8770 - loss: 0.3854 - val_accuracy: 0.3175 - val_loss: 1.5054
Epoch 4/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 158ms/step - accuracy: 0.8849 - loss: 0.2490 - val_accuracy: 0.1429 - val_loss: 1.5531
Epoch 5/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 159ms/step - accuracy: 0.9127 - loss: 0.2948 - val_accuracy: 0.4603 - val_loss: 1.5382
Epoch 6/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 161ms/step - accuracy: 0.9325 - loss: 0.2154 - val_accuracy: 0.3810 - val_loss: 3.8168
Epoch 7/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 163ms/step - accuracy: 0.9802 - loss: 0.0959 - val_accuracy: 0.3651 - val_loss: 5.0958
Epoch 8/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 165ms/step - accuracy: 0.9127 - loss: 0.2851 - val_accuracy: 0

In [9]:
X_test = []
y_test = []

for classe in CLASSES:
    dossier = f'/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/{classe}'
    videos = os.listdir(dossier)
    for video in videos:
        chemin = os.path.join(dossier, video)
        obj = cv2.VideoCapture(chemin)
        nbFrame = int(obj.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, nbFrame - 1, 16, dtype=int)
        frame_video = []
        for nb in indices:
            obj.set(cv2.CAP_PROP_POS_FRAMES, nb)
            ret, frame = obj.read()
            if ret:
                frame = cv2.resize(frame, (112, 112))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame / 255.0
                frame_video.append(frame)
        obj.release()
        if len(frame_video) == 16:
            X_test.append(frame_video)
            y_test.append(classe)

X_test = np.array(X_test, dtype=np.float32)
y_test_encoded = le.transform(y_test)
y_test_cat = to_categorical(y_test_encoded, num_classes=3)

loss, acc = model.evaluate(X_test, y_test_cat)
print(f"Test accuracy : {acc:.2%}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 11s 5s/step - accuracy: 0.9074 - loss: 2.4991
Test accuracy : 90.74%


In [10]:
def annotate_video(video_path, output_path, model, le, num_frames=16, img_size=112):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))


    buffer = []
    label = ""
    confidence = 0.0

    for f in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break


        resized = cv2.resize(frame, (img_size, img_size))
        resized_rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        buffer.append(resized_rgb / 255.0)


        if len(buffer) == num_frames:
            seq = np.array([buffer], dtype=np.float32)
            pred = model.predict(seq, verbose=0)[0]
            label = le.classes_[np.argmax(pred)]
            confidence = np.max(pred)
            buffer = [] 

        if label:
            scale = 1.5
            total_height = len(le.classes_) * 55
            start_y = (height - total_height) // 2
            
            for i, (cls, score) in enumerate(zip(le.classes_, pred)):
                text = f"{cls}: {score:.0%}"
                (text_w, text_h), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
                y = start_y + i * 55 + text_h
                cv2.rectangle(frame, (10, y - text_h - 8), (10 + text_w + 10, y + 8), (0, 0, 0), -1)
                color = (0, 255, 0) if cls == label else (200, 200, 200)
                cv2.putText(frame, text, (15, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"Vidéo annotée sauvegardée : {output_path}")

test_video = f'/kaggle/input/datasets/landshax/testing/testing.mp4'
annotate_video(test_video, '/kaggle/working/output_annote.mp4', model, le)
import os
print(os.path.exists('/kaggle/working/output_annote.mp4'))
print(os.path.getsize('/kaggle/working/output_annote.mp4'), "bytes")

Vidéo annotée sauvegardée : /kaggle/working/output_annote.mp4
True
27213033 bytes


In [11]:
import subprocess
subprocess.run([
    'ffmpeg', '-i', '/kaggle/working/output_annote.mp4',
    '-vcodec', 'libx264', '-crf', '28',
    '/kaggle/working/output_final.avi'
])

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

CompletedProcess(args=['ffmpeg', '-i', '/kaggle/working/output_annote.mp4', '-vcodec', 'libx264', '-crf', '28', '/kaggle/working/output_final.avi'], returncode=0)